In [1]:
if (!requireNamespace("mixOmics", quietly = TRUE)) {
    options(BioC_mirror = "https://bioconductor.org")
    BiocManager::install(version = "3.18", force = TRUE)
    BiocManager::install("mixOmics")
}

library(mixOmics)
library(BiocParallel)
library(svglite)

Loading required package: MASS

Loading required package: lattice

Loading required package: ggplot2


Loaded mixOmics 6.26.0
Thank you for using mixOmics!
Tutorials: http://mixomics.org
Bookdown vignette: https://mixomicsteam.github.io/Bookdown
Questions, issues: Follow the prompts at http://mixomics.org/contact-us
Cite us:  citation('mixOmics')




In [2]:
DATA_DIR <- "/home/mariya/Data/pallas"
MIXOMICS_DIR <- file.path(DATA_DIR, "mixOmics/models_20260625_keepX100")
PREDICTIONS_OUTPUT_DIR <- "/home/mariya/Development/pallas/reports/output/predictions/bimodal"
REPORT_OUTPUT_DIR <- "/home/mariya/Development/pallas/reports/output/evaluation/bimodal"

DATA_DIR <- file.path(DATA_DIR, "mlomix_datasets")

UNLABELED_DATA_DIR <- file.path(DATA_DIR, "unlabeled-bimodal-reduced")
UNLABELED_METH_FILE <- file.path(UNLABELED_DATA_DIR, "features.dnam.csv")
UNLABELED_GEX_FILE <- file.path(UNLABELED_DATA_DIR, "features.gex.csv")

GEX_MODEL_FILE <- file.path(MIXOMICS_DIR, "gex_model.rds")
DNAM_MODEL_FILE <- file.path(MIXOMICS_DIR, "dnam_model.rds")
DIABLO_MODEL_FILE <- file.path(MIXOMICS_DIR, "diablo_model.rds")

# Make output directory if it doesn't exist
if (!dir.exists(PREDICTIONS_OUTPUT_DIR)) {
    dir.create(PREDICTIONS_OUTPUT_DIR, recursive = TRUE)
}

In [3]:
# Reload in a fresh session with
gex_model    <- readRDS(GEX_MODEL_FILE)
dnam_model   <- readRDS(DNAM_MODEL_FILE)
diablo_model <- readRDS(DIABLO_MODEL_FILE)


In [7]:
source("helpers.R")
test_data <- unlabeled_data(UNLABELED_GEX_FILE, UNLABELED_METH_FILE)

GEX features: 199 
DNAm features: 6582 


In [9]:
X_gex_test <- test_data$X_gex
X_dnam_test <- test_data$X_dnam

head(X_gex_test)
head(X_dnam_test)

,ENSG00000079102,ENSG00000175505,ENSG00000185917,ENSG00000145113,ENSG00000102098,ENSG00000248329,ENSG00000164853,ENSG00000173825,ENSG00000073792,ENSG00000067177,⋯,ENSG00000164741,ENSG00000167528,ENSG00000114757,ENSG00000189325,ENSG00000132780,ENSG00000106004,ENSG00000175591,ENSG00000188269,ENSG00000142599,ENSG00000106665
ALL_1001,-1.5827661,-0.1460320,2.911237,-1.5299684,3.457421,-1.582766,-1.582766,2.1647945,2.539270,0.2604124,⋯,1.1168812,3.501684,-1.582766,-1.316203,7.615305,-1.0412013,-0.7174831,-1.5827661,4.579792,7.013319
ALL_1006,-1.5827661,-0.7228762,3.524019,-1.5489382,6.174623,-1.582766,-1.582766,0.9958399,3.068828,-0.7473307,⋯,0.7062212,2.516275,-1.582766,-1.582766,6.688420,0.8365815,-0.8734115,-1.5827661,4.427494,5.843454
ALL_1034,-1.5248777,-0.4807906,4.187927,-1.5353974,4.486938,-1.316524,-1.582766,2.2287674,2.305732,-0.9488385,⋯,2.0718845,3.168230,-1.582766,-1.582766,6.478231,-1.5827661,0.7675948,-1.5827661,4.923694,6.793327
ALL_1089,-0.4643153,1.0015794,2.261712,1.6751438,4.200666,-1.582766,-1.582766,2.6603496,3.574641,1.4396644,⋯,2.0387824,2.313407,-1.582766,-1.382984,7.109462,1.6399011,-1.5827661,-0.3681951,4.535575,6.476001
ALL_219,-1.5827661,3.4474245,4.453691,0.9322236,6.004418,-1.582766,-1.582766,3.5771946,6.288742,1.7895142,⋯,2.5870597,3.019073,-1.582766,-1.582766,6.055022,3.5065864,-1.4465617,-1.5827661,5.043425,6.692534
ALL_232,2.0749894,2.4775575,4.513134,4.3217408,4.814263,-1.153496,-1.582766,3.9621084,7.171068,3.8012675,⋯,3.2825212,3.253408,-1.582766,2.229840,6.946529,-0.8193227,2.4735901,0.1000447,6.413540,7.483233


,cg17259886,cg13293524,cg16288865,cg12923179,cg11622162,cg07232063,cg03763508,cg18638389,cg02707071,cg08166720,⋯,cg19557110,cg08911692,cg05807885,cg00527174,cg08473752,cg08870143,cg06814191,cg06710207,cg08523727,cg06671690
ALL_1001,0.9516,0.9814,0.9169,0.9049,0.0812,0.8589,0.9422,0.9010,0.9406,0.9370,⋯,0.9510,0.9284,0.1309,0.0638,0.9324,0.9331,0.8939,0.7346,0.8849,0.2948
ALL_1006,0.9559,0.9797,0.9412,0.9483,0.0578,0.9398,0.9619,0.9352,0.1296,0.6786,⋯,0.9668,0.9461,0.0279,0.1654,0.9531,0.9584,0.9381,0.9484,0.9370,0.0106
ALL_1034,0.9540,0.9741,0.6628,0.8811,0.0488,0.9372,0.6771,0.9276,0.9429,0.9502,⋯,0.9542,0.9550,0.4315,0.5411,0.9456,0.8715,0.9197,0.9278,0.9125,0.2242
ALL_1089,0.9498,0.9730,0.9114,0.8628,0.0580,0.6835,0.9546,0.9157,0.9311,0.9354,⋯,0.9353,0.9210,0.0461,0.3776,0.9106,0.9406,0.9222,0.9198,0.9284,0.0164
ALL_219,0.9513,0.9763,0.9206,0.9173,0.1234,0.8236,0.9268,0.9392,0.9407,0.9501,⋯,0.9370,0.9370,0.2616,0.0423,0.8912,0.8474,0.9355,0.8993,0.9145,0.0822
ALL_232,0.9562,0.9729,0.7884,0.9328,0.0736,0.9382,0.9465,0.8097,0.9617,0.9612,⋯,0.9656,0.9520,0.4327,0.1736,0.9526,0.9094,0.9379,0.6422,0.9119,0.0178


In [10]:
# X_gex_test  <- X_gex_test[,  colnames(gex_model$X)]
# X_dnam_test <- X_dnam_test[, colnames(dnam_model$X)]

# Verify
cat("GEX names match: ",  all(colnames(X_gex_test)  == colnames(gex_model$X)),  "\n")
cat("DNAm names match:",  all(colnames(X_dnam_test) == colnames(dnam_model$X)), "\n")

# Show number of features in test set vs model
cat("GEX features in test set: ", ncol(X_gex_test), " vs model: ", ncol(gex_model$X), "\n")
cat("DNAm features in test set: ", ncol(X_dnam_test), " vs model: ", ncol(dnam_model$X), "\n")

GEX names match:  TRUE 
DNAm names match: TRUE 
GEX features in test set:  199  vs model:  199 


DNAm features in test set:  6582  vs model:  6582 


In [11]:

gex_pred    <- predict(gex_model,    X_gex_test,  dist = "max.dist")
dnam_pred   <- predict(dnam_model,   X_dnam_test, dist = "max.dist")
diablo_pred <- predict(diablo_model, newdata = list(gex = X_gex_test,
                                                     dnam = X_dnam_test),
                                                     dist = "max.dist")

In [12]:
# Extract predicted classes at final component
ncomp_gex_final    <- gex_model$ncomp
ncomp_dnam_final   <- dnam_model$ncomp
ncomp_diablo_final <- diablo_model$ncomp

gex_class    <- gex_pred$class$max.dist[,    ncomp_gex_final]
dnam_class   <- dnam_pred$class$max.dist[,   ncomp_dnam_final]
diablo_class <- diablo_pred$WeightedVote$max.dist[, ncol(diablo_pred$WeightedVote$max.dist)]

In [15]:
# Make predictiosn dir if not exist
if (!dir.exists(PREDICTIONS_OUTPUT_DIR)) {
    dir.create(PREDICTIONS_OUTPUT_DIR, recursive = TRUE)
}

# After true_label, create three cols: gex_pred, dnam_pred, diablo_pred
# that contain the prediction from the MAX_COMP component for each model
# Keep the rest of the columns as well
predictions_df <- data.frame(
    "Sample" = rownames(X_gex_test),
    "GEX sPLS-DA prediction" = gex_class,
    "DNAm sPLS-DA prediction" = dnam_class,
    "DIABLO sPLS-DA prediction" = diablo_class,
    "GEX" = gex_pred$class,
    "DNAm" = dnam_pred$class,
    "DIABLO" = diablo_pred$class,
    check.names = FALSE
)

predictions_df <- predictions_df[order(predictions_df$Sample), ]

write.csv(predictions_df, file.path(PREDICTIONS_OUTPUT_DIR, "unlabeled_predictions.csv"), row.names = FALSE)

In [17]:
# Build results data frame
results_df <- data.frame(
  sample_id   = rownames(X_gex_test),
  gex_pred    = as.character(gex_class),
  dnam_pred   = as.character(dnam_class),
  diablo_pred = as.character(diablo_class),
  row.names   = NULL
)

head(results_df)

,sample_id,gex_pred,dnam_pred,diablo_pred
,<chr>,<chr>,<chr>,<chr>
1,ALL_1001,"B-ALL, PAX5alt","B-ALL, PAX5alt","B-ALL, PAX5alt"
2,ALL_1006,"B-ALL, high hyperdiploid","B-ALL, high hyperdiploid","B-ALL, high hyperdiploid"
3,ALL_1034,"B-ALL, iAMP21","B-ALL, iAMP21","B-ALL, iAMP21"
4,ALL_1089,"B-ALL, PAX5alt","B-ALL, PAX5alt","B-ALL, PAX5alt"
5,ALL_219,"AML, NPM1alt","B-ALL, BCR::ABL1-like","B-ALL, high hyperdiploid"
6,ALL_232,"B-ALL, BCR::ABL1-like","B-ALL, BCR::ABL1-like","B-ALL, BCR::ABL1-like"
